## Libraries

In [14]:
import numpy as np
import pandas as pd
from statsmodels.tsa.seasonal import STL
from sklearn.preprocessing import StandardScaler
from sklearn.kernel_approximation import RBFSampler
from sklearn.linear_model import SGDRegressor
from scipy.stats import norm
from statsmodels.stats.diagnostic import acorr_ljungbox
from sklearn.neighbors import NearestNeighbors
from sklearn.base import BaseEstimator, TransformerMixin
from tqdm import tqdm

## Config

In [15]:
TIME_COL   = "Date"
TARGET_COL = "AveragePrice"
ENTITY_COL = "AreaCode"

TRAIN_END_DATE  = pd.Timestamp("2022-03-31")
TEST_START_DATE = pd.Timestamp("2022-04-01")

# ---- INSERT YOUR BEST LAG SET & PARAMS HERE ----
# These should come from your tuning results_df.iloc[0]["params"] + lag_set
best_lag_set = [1, 2, 3, 12]  # example: replace with your selected lag set

best_kernel_params = {
    "temporal_gammas": (0.05,0.1),
    "temporal_n_components": 150,
    "spatial_gamma": 0.05,
    "spatial_n_components": 150,
}

best_sgd_params = {
    "alpha": 1e-6,
    "epsilon": 0.1,
    # keep solver knobs aligned to tuning code
    "learning_rate": "invscaling",
    "eta0": 0.01,
    "max_iter": 3000,
    "tol": 1e-3,
    "random_state": 42,
}

# ---- FEATURES (must match your dataset column names) ----
continuous_cols = [
    "AvgNeighbourPrice_lag1",
    "local_I_lag1",
    "area_km2",
    "centroid_x",
    "centroid_y",
    "CoL_distance_km",
    "LA_FE",
    "sdlt_perc_threshold",
    "dwelling_stock_per_1000",
    "population",
    "ashe_weekly",
    "base_rate",
    "claimant_count_prop",
    "planning_decisions_per_1000",
    "planning_granted_prop",
    "rail_station_entry_exit",
    "GDP",
    "CPIH",
    "LA_embed_0",
    "LA_embed_1",
    "LA_embed_2",
    "LA_embed_3",
    "LA_embed_4",
]

categorical_cols = [
    "LMIQuadrantlag1_2.0",
    "LMIQuadrantlag1_3.0",
    "LMIQuadrantlag1_4.0",
    "Region_East of England",
    "Region_London",
    "Region_North East",
    "Region_North West",
    "Region_South East",
    "Region_South West",
    "Region_West Midlands",
    "Region_Yorkshire and The Humber",
]

# Block definitions (same as tuning)
spatial_cols = ["area_km2", "centroid_x", "centroid_y", "CoL_distance_km"]
other_continuous_cols = [c for c in continuous_cols if c not in spatial_cols]
other_cols = other_continuous_cols + categorical_cols


## Metric functions

In [16]:
def mae(y,yhat): return np.mean(np.abs(y-yhat))
def rmse(y,yhat): return np.sqrt(np.mean((y-yhat)**2))
def smape(y,yhat,eps=1e-8):
    return 100*np.mean(2*np.abs(yhat-y)/(np.abs(y)+np.abs(yhat)+eps))

def mase(y,yhat,y_train,m=12,eps=1e-8):
    naive = np.abs(y_train[m:] - y_train[:-m])
    return np.mean(np.abs(y-yhat)) / (np.mean(naive)+eps)

def directional_accuracy(df,entity,time,y,yhat):
    def f(x):
        return np.mean(np.sign(x[y].diff()) == np.sign(x[yhat].diff()))
    return df.groupby(entity).apply(f).mean()

def growth_rate_error(df,entity,time,y,yhat,m=12):
    def f(x):
        return np.mean(np.abs((x[y].pct_change(m) - x[yhat].pct_change(m))))
    return df.groupby(entity).apply(f).mean()

def morans_i(residuals,xs,ys,k=5):
    N=len(residuals)
    X=residuals-np.mean(residuals)
    coords=np.column_stack([xs,ys])
    nbrs=NearestNeighbors(n_neighbors=k+1).fit(coords)
    _,idx=nbrs.kneighbors(coords)
    W=np.zeros((N,N))
    for i in range(N):
        W[i,idx[i][1:]]=1
    W=W/np.sum(W,axis=1,keepdims=True)
    num=np.sum(W*(X[:,None]*X[None,:]))
    den=np.sum(X**2)
    return (N/np.sum(W))*num/den

def crps_gaussian(y,mu,sigma,eps=1e-8):
    a=(y-mu)/(sigma+eps)
    return np.mean(sigma*(1/np.sqrt(np.pi)-2*norm.pdf(a)-a*(2*norm.cdf(a)-1)))

## Multi kernel

In [17]:
class MultiRBFFeatures(BaseEstimator, TransformerMixin):
    """Concatenate multiple RBFSampler maps (multi-scale RBF)."""
    def __init__(self, gammas=(0.05, 0.1, 0.2), n_components=200, random_state=42):
        self.gammas = tuple(gammas)
        self.n_components = int(n_components)
        self.random_state = int(random_state)

    def fit(self, X, y=None):
        self.samplers_ = []
        for i, g in enumerate(self.gammas):
            s = RBFSampler(
                gamma=float(g),
                n_components=self.n_components,
                random_state=self.random_state + i
            )
            s.fit(X)
            self.samplers_.append(s)
        return self

    def transform(self, X):
        return np.hstack([s.transform(X) for s in self.samplers_])


## Rolling STL feature builder

In [18]:
def add_rolling_stl_components(
    df,
    entity_col,
    time_col,
    target_col,
    period=12,
    min_history=24,
    window=120,        # set None for expanding, or e.g. 120 to match your 10y window
    robust=True,
    show_progress=True,
):
    """
    For each LA, compute STL components at time t using only y up to time t.
    We assign the *last* STL values from the fitted history to that time t.

    IMPORTANT:
    - This creates stl_trend/stl_seasonal/stl_resid for each row.
    - You should only use *lags* of these components (e.g., lag1/lag12/lag24)
      when predicting y_t, otherwise you'd leak y_t into its own features.
    """
    df = df.sort_values([entity_col, time_col]).copy()
    df["stl_trend"] = np.nan
    df["stl_seasonal"] = np.nan
    df["stl_resid"] = np.nan

    # Precompute total iterations for tqdm
    groups = list(df.groupby(entity_col))
    total_steps = sum(len(sub) for _, sub in groups)

    iterator = tqdm(
        groups,
        desc="Rolling STL per LA",
        total=len(groups),
        leave=True,
        disable=not show_progress,
    )

    for la, sub in iterator:
        sub = sub.sort_values(time_col)
        y = sub[target_col].astype(float).values
        n = len(sub)

        for t in range(n):
            start = 0 if window is None else max(0, t - window + 1)
            hist = y[start : t + 1]

            if len(hist) < min_history or np.isnan(hist).any():
                continue

            try:
                res = STL(hist, period=period, robust=robust).fit()

                idx = sub.index[t]
                df.loc[idx, "stl_trend"]    = res.trend[-1]
                df.loc[idx, "stl_seasonal"] = res.seasonal[-1]
                df.loc[idx, "stl_resid"]    = res.resid[-1]

            except Exception:
                continue

    return df

## Load data

In [19]:
df = pd.read_excel("../../data/full_data.xlsx", parse_dates=[TIME_COL])
df = df.query('Date < "2024-03-31"')

df = df.sort_values([ENTITY_COL,TIME_COL]).reset_index(drop=True)

df_train_all = df[df[TIME_COL] <= TRAIN_END_DATE].copy()
df_test      = df[df[TIME_COL] >= TEST_START_DATE].copy()

## Training

In [20]:
combined = pd.concat([df_train_all, df_test], axis=0).sort_values([ENTITY_COL, TIME_COL]).copy()
combined = add_rolling_stl_components(
    combined,
    entity_col=ENTITY_COL,
    time_col=TIME_COL,
    target_col=TARGET_COL,
    period=12,
    min_history=24,
    robust=True,
)

# Create selected lag columns
for lag in best_lag_set:
    for comp in ["stl_trend", "stl_seasonal", "stl_resid"]:
        combined[f"{comp}_lag{lag}"] = combined.groupby(ENTITY_COL)[comp].shift(lag)

lag_cols = [f"{c}_lag{l}" for c in ["stl_trend", "stl_seasonal", "stl_resid"] for l in best_lag_set]
feature_cols = continuous_cols + categorical_cols + lag_cols

df_train = combined[combined[TIME_COL] <= TRAIN_END_DATE].dropna(subset=feature_cols).copy()
df_test  = combined[combined[TIME_COL] >= TEST_START_DATE].dropna(subset=feature_cols).copy()

# =========================================================
# Build block matrices
# =========================================================
y_train_raw = df_train[TARGET_COL].values.reshape(-1, 1)
y_test_true = df_test[TARGET_COL].values

# Temporal block = STL lags only
X_train_temp = df_train[lag_cols].copy()
X_test_temp  = df_test[lag_cols].copy()

# Spatial block
X_train_spat = df_train[spatial_cols].copy()
X_test_spat  = df_test[spatial_cols].copy()

# Other block = remaining continuous + categoricals (linear)
X_train_other = df_train[other_cols].copy()
X_test_other  = df_test[other_cols].copy()

# =========================================================
# Scale blocks (train only)
# =========================================================
temp_scaler = StandardScaler()
X_train_temp = temp_scaler.fit_transform(X_train_temp)
X_test_temp  = temp_scaler.transform(X_test_temp)

spat_scaler = StandardScaler()
X_train_spat = spat_scaler.fit_transform(X_train_spat)
X_test_spat  = spat_scaler.transform(X_test_spat)

other_scaler = StandardScaler()
X_train_other_scaled = X_train_other.copy()
X_test_other_scaled  = X_test_other.copy()

X_train_other_scaled[other_continuous_cols] = other_scaler.fit_transform(
    X_train_other_scaled[other_continuous_cols]
)
X_test_other_scaled[other_continuous_cols] = other_scaler.transform(
    X_test_other_scaled[other_continuous_cols]
)

Z_train_other = X_train_other_scaled.values
Z_test_other  = X_test_other_scaled.values

# Scale y (train only)
y_scaler = StandardScaler()
y_train_scaled = y_scaler.fit_transform(y_train_raw).ravel()

# =========================================================
# Kernel maps (FINAL selected)
# =========================================================
temporal_map = MultiRBFFeatures(
    gammas=best_kernel_params["temporal_gammas"],
    n_components=best_kernel_params["temporal_n_components"],
    random_state=42,
)
Z_train_temp = temporal_map.fit_transform(X_train_temp)
Z_test_temp  = temporal_map.transform(X_test_temp)

spatial_map = RBFSampler(
    gamma=best_kernel_params["spatial_gamma"],
    n_components=best_kernel_params["spatial_n_components"],
    random_state=123,
)
Z_train_spat = spatial_map.fit_transform(X_train_spat)
Z_test_spat  = spatial_map.transform(X_test_spat)

# Concatenate into Z
Z_train = np.hstack([Z_train_temp, Z_train_spat, Z_train_other])
Z_test  = np.hstack([Z_test_temp,  Z_test_spat,  Z_test_other])

# Scale Z (train only) — matches tuning setup for SGD stability
Z_scaler = StandardScaler()
Z_train = Z_scaler.fit_transform(Z_train)
Z_test  = Z_scaler.transform(Z_test)




Rolling STL per LA: 100%|██████████| 294/294 [05:12<00:00,  1.06s/it]


## Final fit

In [21]:
model = SGDRegressor(
    loss="epsilon_insensitive",
    penalty="l2",
    epsilon=best_sgd_params["epsilon"],
    alpha=best_sgd_params["alpha"],
    learning_rate=best_sgd_params.get("learning_rate", "invscaling"),
    eta0=best_sgd_params.get("eta0", 0.01),
    max_iter=best_sgd_params.get("max_iter", 3000),
    tol=best_sgd_params.get("tol", 1e-3),
    random_state=best_sgd_params.get("random_state", 42),
)
model.fit(Z_train, y_train_scaled)

y_pred_scaled = model.predict(Z_test)
y_pred = y_scaler.inverse_transform(y_pred_scaled.reshape(-1, 1)).ravel()

df_test = df_test.copy()
df_test["y_pred"] = y_pred
df_test["resid"] = y_test_true - y_pred

## Evaluation

In [22]:
# =========================================================
# Uncertainty (approximate)
# With SGD we don't have a closed-form posterior like Ridge.
# We'll use a simple homoskedastic Gaussian approximation:
# sigma_hat from TRAIN residuals in original £ scale.
# =========================================================
train_pred_scaled = model.predict(Z_train)
train_pred = y_scaler.inverse_transform(train_pred_scaled.reshape(-1, 1)).ravel()
train_true = y_train_raw.ravel()

sigma_hat = float(np.std(train_true - train_pred, ddof=1))
y_std = np.full_like(y_pred, sigma_hat, dtype=float)

z = 1.96
y_lower = y_pred - z * y_std
y_upper = y_pred + z * y_std

PICP = float(np.mean((y_test_true >= y_lower) & (y_test_true <= y_upper)))
PIW  = float(np.mean(y_upper - y_lower))
CRPS = crps_gaussian(y_test_true, y_pred, y_std)

# =========================================================
# Global metrics
# =========================================================
global_mae   = mae(y_test_true, y_pred)
global_rmse  = rmse(y_test_true, y_pred)
global_smape = smape(y_test_true, y_pred)
global_mase  = mase(y_test_true, y_pred, train_true)

# =========================================================
# Across-LA consistency
# =========================================================
la_mae = df_test.groupby(ENTITY_COL).apply(lambda x: mae(x[TARGET_COL].values, x["y_pred"].values))
median_mae = float(np.median(la_mae))
p75_mae    = float(np.percentile(la_mae, 75))

# =========================================================
# Spatio-temporal diagnostics
# =========================================================
la_resid = df_test.groupby(ENTITY_COL)["resid"].mean()

centroids = (
    df_test.drop_duplicates(ENTITY_COL)
           .set_index(ENTITY_COL)[["centroid_x", "centroid_y"]]
           .loc[la_resid.index]
)

I_moran = morans_i(
    la_resid.values,
    centroids["centroid_x"].values,
    centroids["centroid_y"].values
)

monthly_resid = df_test.groupby(TIME_COL)["resid"].mean()
lb_res = acorr_ljungbox(monthly_resid, lags=[12], return_df=True)
q_stat = float(lb_res["lb_stat"].iloc[0])
p_val  = float(lb_res["lb_pvalue"].iloc[0])

# =========================================================
# Direction & growth
# =========================================================
dir_acc = directional_accuracy(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred")
gre_mae = growth_rate_error(df_test, ENTITY_COL, TIME_COL, TARGET_COL, "y_pred")


C:\Users\slong\AppData\Local\Temp\ipykernel_27772\2671876908.py:33: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  la_mae = df_test.groupby(ENTITY_COL).apply(lambda x: mae(x[TARGET_COL].values, x["y_pred"].values))
C:\Users\slong\AppData\Local\Temp\ipykernel_27772\1334586382.py:13: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  return df.groupby(entity).apply(f).mean()
C:\Users\slong\AppData\Local\Temp\ipykernel_27772\

## Save results 

In [23]:
summary_df = pd.DataFrame([{
    "model": "SGD_EPS_INSENSITIVE_BLOCKWISE_RFF",
    "train_end": str(TRAIN_END_DATE.date()),
    "test_start": str(TEST_START_DATE.date()),
    "lag_set": str(best_lag_set),
    "kernel_params": str(best_kernel_params),
    "sgd_params": str(best_sgd_params),
    "MAE": global_mae,
    "RMSE": global_rmse,
    "sMAPE": global_smape,
    "MASE": global_mase,
    "Median_LA_MAE": median_mae,
    "P75_LA_MAE": p75_mae,
    "Morans_I": I_moran,
    "LjungBox_Q12": q_stat,
    "LjungBox_p": p_val,
    "Directional_Accuracy": dir_acc,
    "GrowthRateError_MAE": gre_mae,
    "PICP_95": PICP,
    "PIW_95": PIW,
    "CRPS": CRPS,
    "Sigma_hat_train": sigma_hat,
}])

summary_df.to_excel("../../results/final_SVR_results.xlsx", index=False)
la_mae.reset_index().to_excel("../../results/final_SVR_la_mae.xlsx", index=False)

print("\n=== FINAL TEST RESULTS SAVED ===")
print(summary_df.T)


=== FINAL TEST RESULTS SAVED ===
                                                                      0
model                                 SGD_EPS_INSENSITIVE_BLOCKWISE_RFF
train_end                                                    2022-03-31
test_start                                                   2022-04-01
lag_set                                                   [1, 2, 3, 12]
kernel_params         {'temporal_gammas': (0.05, 0.1), 'temporal_n_c...
sgd_params            {'alpha': 1e-06, 'epsilon': 0.1, 'learning_rat...
MAE                                                         53163.43505
RMSE                                                       63235.723558
sMAPE                                                         23.355024
MASE                                                           2.525641
Median_LA_MAE                                              51264.662709
P75_LA_MAE                                                 60538.978028
Morans_I                      

In [24]:
print("TEST rows:", len(df_test))
print("TEST date range:", df_test[TIME_COL].min(), "->", df_test[TIME_COL].max())
print("Unique months:", df_test[TIME_COL].nunique())
print("Unique LAs:", df_test[ENTITY_COL].nunique())

TEST rows: 7056
TEST date range: 2022-04-01 00:00:00 -> 2024-03-01 00:00:00
Unique months: 24
Unique LAs: 294
